# Lecture 09: NLP Fundamentals
## Tokenization, Vocabulary, and Padding with PyTorch

This beginner notebook starts with only six short English sentences.

- We do **not** build a classification model.
- We do **not** define a Python `class`.
- We do **not** define a custom `def` function.
- We use simple loops first, and PyTorch tensors later.
- The final section prepares input-target pairs for future next-token prediction.



## Practice Problem

**Scenario:** Use a new four-sentence corpus about a park and a garden:

```python
corpus_p = [
    "Noah walks a dog in the park.",
    "Mia walks a cat in the park.",
    "Emma feeds a bird in the garden.",
    "Liam feeds a fish in the garden.",
]
```

Using the **same method** as this lecture, do the following:

1. Clean and tokenize the corpus (lowercase, keep the period, remove
   repeated spaces).
2. Build a vocabulary with the same four special tokens
   (`<PAD>`, `<UNK>`, `<BOS>`, `<EOS>`), ordering ordinary tokens by
   frequency and then alphabetically.
3. Encode every sentence with `<BOS>` and `<EOS>` added.
4. Pad the encoded sentences into one batch tensor with
   `pad_sequence`, and compute the attention mask.
5. Encode the new sentence `"Noah feeds a rabbit in the park."` and
   confirm that `"rabbit"` receives the `<UNK>` ID.


### Solution

In [1]:
SEED = 42
torch.manual_seed(SEED)

print("PyTorch version:", torch.__version__)
print("Random seed:", SEED)

NameError: name 'torch' is not defined

In [ ]:
import matplotlib.pyplot as plt

import re
import torch
from torch.nn.utils.rnn import pad_sequence
SEED = 42
torch.manual_seed(SEED)

print("PyTorch version:", torch.__version__)
print("Random seed:", SEED)


# ----- New corpus -----
corpus_p = [
    "Noah walks a dog in the park.",
    "Mia walks a cat in the park.",
    "Emma feeds a bird in the garden.",
    "Liam feeds a fish in the garden.",
]

# ----- Clean -----
cleaned_corpus_p = []
for sentence in corpus_p:
    cleaned = sentence.strip()
    cleaned = re.sub(r"\s+", " ", cleaned)
    cleaned = cleaned.lower()
    cleaned_corpus_p.append(cleaned)

# ----- Tokenize -----
token_pattern_p = r"[a-z]+(?:'[a-z]+)?|[.,!?;]"
tokenized_corpus_p = []
for sentence in cleaned_corpus_p:
    tokens = re.findall(token_pattern_p, sentence)
    tokenized_corpus_p.append(tokens)

for number, tokens in enumerate(tokenized_corpus_p, start=1):
    print(f"Sentence {number}: {tokens}")

# ----- Special tokens -----
special_tokens_p = ["<PAD>", "<UNK>", "<BOS>", "<EOS>"]
PAD_ID_p, UNK_ID_p, BOS_ID_p, EOS_ID_p = 0, 1, 2, 3

# ----- Vocabulary: frequency, then alphabetical -----
token_counts_p = {}
for sentence_tokens in tokenized_corpus_p:
    for token in sentence_tokens:
        if token not in token_counts_p:
            token_counts_p[token] = 0
        token_counts_p[token] += 1

ordered_tokens_p = sorted(token_counts_p.keys())
ordered_tokens_p.sort(key=token_counts_p.get, reverse=True)

vocabulary_tokens_p = special_tokens_p + ordered_tokens_p
token_to_id_p = {}
id_to_token_p = {}
for token_id, token in enumerate(vocabulary_tokens_p):
    token_to_id_p[token] = token_id
    id_to_token_p[token_id] = token

print()
print("Vocabulary size:", len(token_to_id_p))

# ----- Encode -----
encoded_corpus_p = []
for sentence_tokens in tokenized_corpus_p:
    token_ids = [BOS_ID_p]
    for token in sentence_tokens:
        token_ids.append(token_to_id_p.get(token, UNK_ID_p))
    token_ids.append(EOS_ID_p)
    encoded_corpus_p.append(torch.tensor(token_ids, dtype=torch.long))

print()
print("Encoded corpus:")
for token_ids in encoded_corpus_p:
    print(token_ids.tolist())

# ----- Pad -----
padded_ids_p = pad_sequence(encoded_corpus_p, batch_first=True, padding_value=PAD_ID_p)
attention_mask_p = (padded_ids_p != PAD_ID_p).long()

print()
print("Padded token-ID tensor:")
print(padded_ids_p)
print()
print("Attention mask:")
print(attention_mask_p)

# ----- Encode a new sentence with an unseen word -----
new_sentence_p = "Noah feeds a rabbit in the park."
new_cleaned_p = re.sub(r"\s+", " ", new_sentence_p.strip().lower())
new_tokens_p = re.findall(token_pattern_p, new_cleaned_p)
new_ids_p = [BOS_ID_p] + [token_to_id_p.get(t, UNK_ID_p) for t in new_tokens_p] + [EOS_ID_p]

print()
print("New sentence tokens:", new_tokens_p)
print("New sentence IDs:", new_ids_p)
print("Words mapped to <UNK>:", [t for t in new_tokens_p if t not in token_to_id_p])


# ----- Visualization 1: token frequency in the new corpus -----
plt.figure(figsize=(5, 3))
plt.bar(ordered_tokens_p, [token_counts_p[t] for t in ordered_tokens_p])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Count")
plt.title("Practice: Token Frequency")
plt.grid(False)
plt.tight_layout()
plt.show()

# ----- Visualization 2: padded token-ID matrix as a heatmap -----
plt.figure(figsize=(5, 3))
plt.imshow(padded_ids_p.numpy(), aspect="auto", cmap="viridis")
plt.xlabel("Position")
plt.ylabel("Sentence row")
plt.title("Practice: Padded Token IDs")
plt.colorbar(label="Token ID")
plt.grid(False)
plt.tight_layout()
plt.show()
